# Index Building with Llama Stack & Milvus on RHOAI

#### Prerequesites

- run one foundational model on RHOAI
- run one embedding model on RHOAI
- run Llama Stack Server server in version 0.5.x on RHOAI (using remote::vllm providers) with enabled inline Milvus


Add environment variables
- **LLAMASTACK_URL**
- **LLAMASTACK_APIKEY** if auth/authz is enabled on Llama Stack Server

NOTE: You can run also LLama Stack locally, using either remote::vllm (for remote models) or ollama (for local models) providers.


### Install dependencies if needed

In [ ]:
!pip install dotenv
!pip install pypdf
!pip install ai4rag==0.5.0

#### Import dependencies

**NOTE:** This assumes adding environment variables from `.env` file for Llama Stack URL and APIKEY.

**NOTE:** This assumes logging from httpx library on WARNING level.

In [ ]:
import logging
import os

from dotenv import load_dotenv

load_dotenv()

logging.getLogger("httpx").setLevel(logging.WARNING)

#### Create LlamaStackClient object

**NOTE:** If you do not have authorization enabled on your Llama Stack instance, just skip providing `LLAMASTACK_APIKEY`.


In [ ]:
from llama_stack_client import LlamaStackClient

client = LlamaStackClient(
    base_url=os.getenv("LLAMASTACK_URL", "http://localhost:5321"),
    api_key=os.getenv("LLAMASTACK_APIKEY"),
)

#### Models used in the example

In [3]:
EMBEDDING_MODEL = "vllm-embedding/granite-278m-multilingual-1"
EMBEDDING_MODEL_DIMENSION = 768

LLM_MODEL = "vllm-inference-llama-3-1/redhataillama-31-8b-instruct"

In [4]:
client.models.list()[0].model_dump()

{'id': 'vllm-inference-llama-3-1/RedHatAI/Llama-3.1-8B-Instruct',
 'created': 1771842861,
 'owned_by': 'llama_stack',
 'custom_metadata': {'model_type': 'llm',
  'provider_id': 'vllm-inference-llama-3-1',
  'provider_resource_id': 'RedHatAI/Llama-3.1-8B-Instruct'},
 'object': 'model'}

#### Tests model availability via Llama Stack

In [5]:
msgs = [{"role": "user", "content": "Generate a short poem"}]

llm_resp = client.chat.completions.create(messages=msgs, model=LLM_MODEL)

llm_resp.model_dump()

{'id': 'chatcmpl-be08f90f083445dda264f5195b8e9620',
 'choices': [{'finish_reason': 'stop',
   'index': 0,
   'message': {'content': 'Here\'s a short poem:\n\n"Moonlit Night"\n\nThe moon casts its gentle glow,\nA silver light, in the midnight\'s flow.\nThe stars shine bright, like diamonds rare,\nA celestial show, beyond compare.\n\nThe world is hushed, in a peaceful sleep,\nAs the moon\'s soft beams, our dreams do keep.\nIn this quiet hour, I find my peace,\nA sense of calm, that the world can\'t release.',
    'name': None,
    'role': 'assistant',
    'tool_calls': None,
    'refusal': None,
    'annotations': None,
    'audio': None,
    'function_call': None,
    'reasoning': None,
    'reasoning_content': None},
   'logprobs': None,
   'stop_reason': None,
   'token_ids': None}],
 'created': 1771842862,
 'model': 'vllm-inference-llama-3-1/redhataillama-31-8b-instruct',
 'object': 'chat.completion',
 'usage': {'completion_tokens': 90,
  'prompt_tokens': 39,
  'total_tokens': 129,
 

In [6]:
emb_response = client.embeddings.create(input="Hello", model=f"{EMBEDDING_MODEL}")
emb_response.model_dump()

{'data': [{'embedding': [-0.01421553734689951,
    0.02087596245110035,
    -0.00365329347550869,
    0.014414356090128422,
    0.03260626643896103,
    -0.05805505812168121,
    0.028232255950570107,
    0.043143656104803085,
    0.048312943428754807,
    0.003305360907688737,
    0.004423716105520725,
    0.014712583273649216,
    0.008151566609740257,
    -0.007903043180704117,
    0.02027950808405876,
    0.07634638249874115,
    0.07356292009353638,
    -0.05765742436051369,
    -0.030618079006671906,
    -0.002708904678002,
    -0.012078235857188702,
    0.035588547587394714,
    -0.027238162234425545,
    0.07952748239040375,
    0.020478326827287674,
    -0.05368104949593544,
    -0.013917309232056141,
    -0.027039343491196632,
    -0.06401962041854858,
    0.03598618507385254,
    0.012674692086875439,
    0.00835038535296917,
    -0.01640254259109497,
    0.050698768347501755,
    -0.0020875963382422924,
    0.004200045019388199,
    0.010736209340393543,
    0.0328050851821

### Documents for knwoledge database building and benchmarking data

**NOTE:** The below example assumes that both documents for index building and benchmarking data resides in subfolder: `data/`. Moreover, this example itself does not deliver these artifacts - you need to replace:
- `documents_path` with list of paths to documents used for index building
- `benchmark_qa_path` with filename of benchmarking dataset, against which we'll test retrieval

In [7]:
import json
from pathlib import Path

In [8]:
docs_folder_path = Path("data/")
documents_path = [
    docs_folder_path.joinpath(doc)
    for doc in (
        "ibm-annual-report-2024-pt1_1-20.pdf",
        "ibm-annual-report-2024-pt2_20-40.pdf",
    )
]
benchmark_qa_path = docs_folder_path.joinpath(
    "ibm_annual_report_pdf_benchmarking_data.json"
)

In [9]:
benchmark_qa = json.loads(benchmark_qa_path.read_bytes())
benchmark_qa[:3]

[{'correct_answer_document_ids': ['ibm-annual-report-2024-pt1_1-20.pdf'],
  'question': 'What was the operating other income and expense in 2024?',
  'correct_answer': '$1,656 million'},
 {'correct_answer_document_ids': ['ibm-annual-report-2024-pt1_1-20.pdf'],
  'question': 'What is the amount of gain on land/building dispositions included in "Other"',
  'correct_answer': '$126 million'},
 {'correct_answer_document_ids': ['ibm-annual-report-2024-pt1_1-20.pdf'],
  'question': 'How much was the non-operating retirement-related costs in the current-year period?',
  'correct_answer': '$3,457 million'}]

## Index building

### 1.0 Define helper objects

In [ ]:
import io
from pathlib import Path
from typing import Sequence

from langchain_core.documents import Document
from pypdf import PdfReader


def _txt_to_string(binary_data: bytes) -> str:
    return binary_data.decode("utf-8", errors="ignore")


def _pdf_to_string(binary_data: bytes) -> str:
    with io.BytesIO(binary_data) as open_pdf_file:
        reader = PdfReader(open_pdf_file)
        full_text = [page.extract_text() for page in reader.pages]
        return "\n".join(full_text)


class FileStoreException(Exception):
    pass


class FileStore:
    """
    Class used to load locally saved input files.
    """

    suffix_to_func = {
        ".txt": _txt_to_string,
        ".pdf": _pdf_to_string,
    }

    def __init__(self, path: str | Path | Sequence[str] | Sequence[Path]):
        self.path = Path(path)
        self.is_dir = path.is_dir()
        self.files = {}

    def __repr__(self) -> str:
        ret = f"{self.__class__.__name__}(path={self.path})"
        return ret

    def load_as_documents(self) -> list[Document]:
        """Read files as langchain documents"""
        contents = self._load_content()
        documents = [
            Document(page_content=content[0], metadata={"document_id": content[1]})
            for content in contents
        ]

        return documents

    def _load_content(self) -> list[tuple[str, str]]:
        """Load file(s) from given path"""
        if self.is_dir:
            contents = [
                (self._read_single_content(file), file.name)
                for file in self.path.iterdir()
            ]
            return contents

        return [(self._read_single_content(self.path), self.path.name)]

    def _process_file(self, filepath: Path, file_content: bytes | None) -> str:
        """
        Extracts text from bytes for various file types.

        Parameters
        ----------
        filepath : Path
            Path to file

        file_content : bytes | None
            File content as bytes

        Returns
        -------
        str
            Extracted file text
        """

        handler = self.suffix_to_func.get(filepath.suffix, None)

        try:
            text = handler(file_content)
        except Exception as exc:
            raise FileStoreException("Failed to load file.") from exc

        return text

    def _read_single_content(self, filepath: Path) -> str:
        """Read single file"""
        with open(filepath, "rb") as file:
            content = file.read()
        text = self._process_file(filepath=filepath, file_content=content)
        self.files[str(filepath)] = text
        return text

### 1.1  Load documents in LangChain represenation

In [11]:
file_store = FileStore(Path("data/docs"))
documents = file_store.load_as_documents()

In [12]:
documents

[Document(metadata={'document_id': 'ibm-annual-report-2024-pt1_1-20.pdf'}, page_content="\n\n\nI n f r a s t r u c t u r eC o n s u l t i n gS o f t w a r e\n2\nt o s h a r e h o l d e r s t h r o u g h d i v i d e n d s . I B M e x p a n d e d o p e r a t i n g\ng r o s s p r o f t m a r g i n b y 1 3 0 b a s i s p o i n t s i n 2 0 2 4 , d r i v e n b y a\n\ue006 o c u s o n h i g h - v a l u e o \ue006 \ue006 e r i n g s a n d p r o d u c t i v i t y i n i t i a t i v e s .\nS o f t w a r e r e v e n u e g r e w 9 % a t c o n s t a n t c u r r e n c y , r e f l e c t i n g\ns t r o n g d e m a n d f o r o u r a d v a n c e d c a p a b i l i t i e s i n h y b r i d c l o u d ,\nd a t a a n d A I , a u t o m a t i o n , t r a n s a c t i o n p r o c e s s i n g , a n d s e c u r i t y .\nR e d H a t p e r f o r m e d w e l l a s c l i e n t s e m b r a c e d i t s o p e n h y b r i d\nc l o u d p l a t f o r m .\nC o n s u l t i n g r e v e n u e i n c r e a s e d 1 % a t c o n s t a 

### 1.2 Define Llama Stack embedding model object

In [13]:
from ai4rag.rag.embedding.llama_stack import LSEmbeddingModel

embedding_model = LSEmbeddingModel(
    model_id=EMBEDDING_MODEL,
    client=client,
    params=dict(
        provider_id="milvus",
        embedding_dimension=EMBEDDING_MODEL_DIMENSION,
        context_length=512,
    ),
)

### 1.3 Define Llama Stack Vector Store object

In [14]:
from ai4rag.rag.vector_store.llama_stack import LSVectorStore

vector_store = LSVectorStore(
    embedding_model=embedding_model,
    client=client,
    provider_id="milvus",
)

In [15]:
vector_store._ls_vs.id

'vs_b1ac3edf-6a01-45fb-afaf-d01c5a335d53'

### 1.4 Chunk and embed documents into vector store

In [19]:
from ai4rag.rag.chunking import LangChainChunker

chunking_method = "recursive"
chunk_size = 256
chunk_overlap = 64

chunker = LangChainChunker(
    method=chunking_method, chunk_size=chunk_size, chunk_overlap=chunk_overlap
)
chunked_documents = chunker.split_documents(documents)
vector_store.add_documents(chunked_documents)

### 1.5 Test retrieval

In [17]:
question = benchmark_qa[-1].get("question")
question

'What was the total revenue for year 2024?'

In [18]:
search_response = client.vector_stores.search(
    vector_store_id=vector_store._ls_vs.id,
    query=question,
    max_num_results=5,
)
search_response

VectorStoreSearchResponse(data=[Data(content=[DataContent(text='S o f t w a r e r e v e n u e g r e w 9 % a t c o n s t a n t c u r r e n c y , r e f l e c t i n g\ns t r o n g d e m a n d f o r o u r a d v a n c e d c a p a b i l i t i e s i n h y b r i d c l o u d ,', type='text', chunk_metadata=None, embedding=None, metadata=None)], file_id='', filename='', score=0.6445577144622803, attributes={}), Data(content=[DataContent(text='( 1 )\n2 0 2 2\n( 1 )\nY r . - t o - Y r .\nP e r c e n t /\nM a r g i n\nC h a n g e\nY r . - t o - Y r .\nP e r c e n t C h a n g e\nA d j u s t e d f o r\nC u r r e n c y\nR e v e n u e\nS o f t w a r e $ 2 5 , 0 1 1 $ 2 3 , 6 2 9 5 . 8 % 5 . 9 %', type='text', chunk_metadata=None, embedding=None, metadata=None)], file_id='', filename='', score=0.6332260966300964, attributes={})], search_query=['What was the total revenue for year 2024?'], has_more=False, next_page=None, object='vector_store.search_results.page')